In [1]:
import aerosandbox as asb
import aerosandbox.numpy as anp
import casadi as ca
from termcolor import cprint

# import jax
# import jax.numpy as jnp
from pathlib import Path

from importlib import reload

reload(asb)

<module 'aerosandbox' from 'd:\\miniconda3\\envs\\py12\\Lib\\site-packages\\aerosandbox\\__init__.py'>

In [ ]:
def fun1():
    func_cache = Path("func_cache.casadi")
    if func_cache.exists():
        func = ca.Function.load(func_cache.as_posix())
    else:
        x = ca.GenSX_sym("x", 3)
        y = x
        for _ in range(1000000):
            y = anp.sin(y)
        y = anp.sum(y)
        func = ca.Function("func", [x], [y])
        func.save(func_cache.as_posix())
    return func


func = fun1()

In [ ]:
cprint(func(anp.ones(3)), "green")

0.00519612


In [4]:
N = 10
func_mpi = func.map(N, "openmp")
cprint(func_mpi(anp.ones((3, N))), "yellow")

[[0.00519612, 0.00519612, 0.00519612, 0.00519612, 0.00519612, 0.00519612, 0.00519612, 0.00519612, 0.00519612, 0.00519612]]


In [5]:
def get_aero():
    Au = ca.GenSX_sym("Au", 8)
    Al = ca.GenSX_sym("Al", 8)
    alpha = ca.GenSX.sym("alpha")

    af = asb.KulfanAirfoil(lower_weights=Al, upper_weights=Au)
    aero = af.get_aero_from_neuralfoil(alpha=alpha, Re=3e5, mach=0.03)
    CL = aero["CL"]
    CD = aero["CD"]

    INPUTS = ca.vcat([Au, Al, alpha])
    OUTPUTS = ca.vcat([CL, CD])

    FUNC = ca.Function("FUNC", [INPUTS], [OUTPUTS])
    return FUNC


aero = get_aero()

In [2]:
def func():
    opti = asb.Opti()
    x = opti.variable(init_guess=1.0, n_vars=1)
    p = opti.parameter(value=1.0, n_params=1)
    z = (2 * x - p) ** 2
    opti.minimize(z)
    sol = opti.solve(verbose=False, expand=True, dry_run=True)

    func = opti.to_function("opti", [p], [x])
    return func


fun = func()
# N = 5
# opti_mpi = opti.map(N, "openmp")

# X = anp.full((2, N), 2)

# cprint(opti_mpi(X), "green")

opti = asb.Opti()
x = opti.variable(init_guess=1, n_vars=1)
p = fun(x)


# print(p)

opti.minimize(x**2 + p)
sol = opti.solve(expand=True)
cprint(sol(x))

This is Ipopt version 3.14.11, running with linear solver MUMPS 5.4.1.

Number of nonzeros in equality constraint Jacobian...:        0
Number of nonzeros in inequality constraint Jacobian.:        0
Number of nonzeros in Lagrangian Hessian.............:        1

Total number of variables............................:        1
                     variables with only lower bounds:        0
                variables with lower and upper bounds:        0
                     variables with only upper bounds:        0
Total number of equality constraints.................:        0
Total number of inequality constraints...............:        0
        inequality constraints with only lower bounds:        0
   inequality constraints with lower and upper bounds:        0
        inequality constraints with only upper bounds:        0

iter    objective    inf_pr   inf_du lg(mu)  ||d||  lg(rg) alpha_du alpha_pr  ls
   0  1.5000000e+00 0.00e+00 2.50e+00   0.0 0.00e+00    -  0.00e+00 0.00e+00 

In [ ]:
opti = asb.Opti()
x = opti.variable(init_guess=1.0, n_vars=5)
obj = anp.sum((x - 10) ** 2)
p = anp.array([[1, 2, 3, 4, 5], [1, 2, 3, 4, 5]]).T

y = ca.hcat([x, p])
opti.minimize(obj)
sol = opti.solve(expand=True)
cprint(sol(y), "green")

This is Ipopt version 3.14.11, running with linear solver MUMPS 5.4.1.

Number of nonzeros in equality constraint Jacobian...:        0
Number of nonzeros in inequality constraint Jacobian.:        0
Number of nonzeros in Lagrangian Hessian.............:        5

Total number of variables............................:        5
                     variables with only lower bounds:        0
                variables with lower and upper bounds:        0
                     variables with only upper bounds:        0
Total number of equality constraints.................:        0
Total number of inequality constraints...............:        0
        inequality constraints with only lower bounds:        0
   inequality constraints with lower and upper bounds:        0
        inequality constraints with only upper bounds:        0

iter    objective    inf_pr   inf_du lg(mu)  ||d||  lg(rg) alpha_du alpha_pr  ls
   0  4.0500000e+02 0.00e+00 1.80e+01   0.0 0.00e+00    -  0.00e+00 0.00e+00 

In [48]:
def gen_fun():
    x = ca.GenSX_sym("x")
    y = x**2
    func = ca.Function("func", [x], [y])
    return func


func = gen_fun()
func_mpi = func.map(5, "openmp")

opti = asb.Opti()
x = opti.variable(init_guess=1.0, n_vars=5)
y = func_mpi(x.T)

for i in range(5):
    opti.subject_to(x[i] + 1 == y[i])

sol = opti.solve()
cprint(sol(x), "green")

This is Ipopt version 3.14.11, running with linear solver MUMPS 5.4.1.

Number of nonzeros in equality constraint Jacobian...:        5
Number of nonzeros in inequality constraint Jacobian.:        0
Number of nonzeros in Lagrangian Hessian.............:        5

Total number of variables............................:        5
                     variables with only lower bounds:        0
                variables with lower and upper bounds:        0
                     variables with only upper bounds:        0
Total number of equality constraints.................:        5
Total number of inequality constraints...............:        0
        inequality constraints with only lower bounds:        0
   inequality constraints with lower and upper bounds:        0
        inequality constraints with only upper bounds:        0

iter    objective    inf_pr   inf_du lg(mu)  ||d||  lg(rg) alpha_du alpha_pr  ls
   0  0.0000000e+00 1.00e+00 0.00e+00   0.0 0.00e+00    -  0.00e+00 0.00e+00 

In [17]:
def gen_opti():
    # opti = asb.Opti()

    # varibles
    Va = ca.GenMX_sym("Va")
    Vt = ca.GenMX_sym("Vt")

    # parameters
    r = ca.GenMX_sym("r")
    R = ca.GenMX_sym("R")
    Rn = ca.GenMX_sym("Rn")
    V0 = ca.GenMX_sym("V0")
    ns = ca.GenMX_sym("ns")
    rho = ca.GenMX_sym("rho")
    sos = ca.GenMX_sym("sos")
    viscosity = ca.GenMX_sym("viscosity")
    theta = ca.GenMX_sym("theta")
    b = ca.GenMX_sym("b")
    Au = ca.GenMX_sym("Au", 8)
    Al = ca.GenMX_sym("Al", 8)
    Le = ca.GenMX_sym("Le")
    Nb = ca.GenMX_sym("Nb")

    # compute
    V_tau = 2 * anp.pi * ns * r
    fai0 = anp.arctan(V0 / V_tau)
    fai = anp.arctan((V0 + Va) / (V_tau - Vt))
    beta = fai - fai0

    # Momentum Theorem
    dT_dr_Momentum = 4 * anp.pi * r * rho * (V0 + Va) * Va
    dM_dr_Momentum = 4 * anp.pi * r**2 * rho * (V0 + Va) * Vt

    # blade element
    W = anp.sqrt((V0 + Va) ** 2 + (V_tau - Vt) ** 2)
    alpha = theta - fai
    Ma = W / sos
    Re = rho * W * b / viscosity

    af = asb.KulfanAirfoil(lower_weights=Al, upper_weights=Au, leading_edge_weight=Le, TE_thickness=0.0)
    # af = asb.KulfanAirfoil("naca0012").set_TE_thickness(0.0)
    aero = af.get_aero_from_neuralfoil(alpha=alpha, Re=Re, mach=Ma, model_size="xxxlarge")
    CL = aero["CL"]
    CD = aero["CD"]
    # CM=aero["CM"]

    q = 0.5 * rho * W**2 * b
    dL_dr = q * CL
    dD_dr = q * CD
    dT_dr_blade = dL_dr * anp.cos(fai) - dD_dr * anp.sin(fai)
    dF_dr_blade = dL_dr * anp.sin(fai) + dD_dr * anp.cos(fai)
    dM_dr_blade = dF_dr_blade * r

    # correct coeffience
    Ft = 2 / anp.pi * anp.arccos(anp.exp(-(Nb * (R - r)) / (2 * r * anp.sin(fai))))
    Fr = 2 / anp.pi * anp.arccos(anp.exp(-(Nb * (r - Rn) / (2 * r * anp.sin(fai)))))
    F = Ft * Fr
    dT_dr_Momentum_correct = dT_dr_Momentum * F
    dM_dr_Momentum_correct = dM_dr_Momentum * F

    # opti.subject_to([dT_dr_blade == dT_dr_Momentum_correct, dM_dr_blade == dM_dr_Momentum_correct])

    # opti.solve(verbose=False, dry_run=True)
    INPUTS = ca.vcat([Va, Vt, r, R, Rn, V0, ns, rho, sos, viscosity, theta, b, Nb, Al, Au, Le])
    OUTPUS = ca.vcat(
        [
            dT_dr_blade,
            dM_dr_blade,
            dT_dr_Momentum_correct,
            dM_dr_Momentum_correct,
            alpha,
            beta,
            CL,
            CD,
            fai,
            V_tau,
            V0,
            Re,
            Ma,
            W,
            Va,
            Vt,
        ]
    )

    # FUNC_blade = opti.to_function("FUNC_blade", [INPUTS], [OUTPUS])
    FUNC_blade = ca.Function("FUNC_blade", [INPUTS], [OUTPUS])
    return FUNC_blade


FUNC_blade = gen_opti()

r = 2.9
R = 3.0
Rn = 0.0
V0 = 0.0
ns = 300 * 2 * anp.pi / 60
ns = ns / (2 * anp.pi)
atmos = asb.Atmosphere(altitude=0.0)
rho = atmos.density()
sos = atmos.speed_of_sound()
viscosity = atmos.dynamic_viscosity()
theta = anp.deg2rad(20)
b = 0.453
Nb = 3
af = asb.KulfanAirfoil("naca0012").set_TE_thickness(0.0)
Al = af.lower_weights
Au = af.upper_weights
Le = af.leading_edge_weight

opti = asb.Opti()
Va = opti.variable(init_guess=1.0, n_vars=1)
Vt = opti.variable(init_guess=1.0, n_vars=1)

INPUTS = ca.vcat([Va, Vt, r, R, Rn, V0, ns, rho, sos, viscosity, theta, b, Nb, Al, Au, Le])
aero = FUNC_blade(INPUTS)

opti.subject_to([aero[0] == aero[2], aero[1] == aero[3]])

sol = opti.solve()

cprint(sol(aero), "green")

This is Ipopt version 3.14.11, running with linear solver MUMPS 5.4.1.

Number of nonzeros in equality constraint Jacobian...:        4
Number of nonzeros in inequality constraint Jacobian.:        0
Number of nonzeros in Lagrangian Hessian.............:        3

Total number of variables............................:        2
                     variables with only lower bounds:        0
                variables with lower and upper bounds:        0
                     variables with only upper bounds:        0
Total number of equality constraints.................:        2
Total number of inequality constraints...............:        0
        inequality constraints with only lower bounds:        0
   inequality constraints with lower and upper bounds:        0
        inequality constraints with only upper bounds:        0

iter    objective    inf_pr   inf_du lg(mu)  ||d||  lg(rg) alpha_du alpha_pr  ls
   0  0.0000000e+00 9.28e+01 0.00e+00   0.0 0.00e+00    -  0.00e+00 0.00e+00 